In [2]:
# Import packages
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import yfinance as yf
from tqdm import tqdm

In [65]:
nflx_chain  = yf.download("NFLX", start="2025-04-02", end="2025-10-02")
spot_chain = yf.download("SPOT", start="2025-04-02", end="2025-10-02")
walt_chain = yf.download("DIS", start="2025-04-02", end="2025-10-02")

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [66]:
len(nflx_chain)


126

In [82]:
n_samples = 100
dt = 1/126
n = 126
n_particles = 500
nflx_close = nflx_chain['Close']
spot_close = spot_chain['Close']
walt_close = walt_chain['Close']


In [83]:
S = nflx_close 
returns = S.pct_change().dropna()
mu0 = returns.mean().item()
print(f"Initial mu estimate: {mu0}")
theta0 = returns.var().item()
print(f"Initial theta estimate: {theta0}")
kappa0 = 2.0 # assume mean-reversion speed
sigma0 = 0.3 # assume vol-of-vol
rho0 = -0.7  

sigma_prior_eta = 0.001
mu_prior_eta = 1.00125
tau_prior_eta = 1/sigma_prior_eta**2
lambda_prior = np.array([[500, 0], [0, 500]]).reshape(2, 2) #2x2
mu_prior = np.array([[35e-6], [0.988]]) # 2x1
a_prior_sigma = 149
b_prior_sigma = 0.025

mu_prior_phi = -0.45
sigma_prior_phi = 0.3
tau_prior_phi = 1/sigma_prior_phi**2
a_prior_omega = 1.03
b_prior_omega = 0.05


Initial mu estimate: 0.001978191252677619
Initial theta estimate: 0.00036684912433881034


In [84]:
def convertCDF(v, V_sort, W_sort, n_particles):

    W_sort = W_sort / np.sum(W_sort)
    N = len(V_sort)

    if v < V_sort[0]:
        return 0.0

    if v > V_sort[-1]:
        return 1.0

    j = np.searchsorted(V_sort, v) - 1
    j = max(0, min(j, N-2))

    vj = V_sort[j]
    vj1 = V_sort[j+1]

    if abs(vj1 - vj) < 1e-15:
        interp_factor = 0.0
    else:
        interp_factor = (v - vj) / (vj1 - vj)

    if j == 0:  # first interval
        weight = W_sort[0] + 0.5 * W_sort[1]
        return interp_factor * weight

    elif j == N-2:  # last interval
        base = np.sum(W_sort[:N-2]) + 0.5 * W_sort[N-2]
        weight = 0.5 * W_sort[N-2] + W_sort[N-1]
        return base + interp_factor * weight

    else:  # middle intervals
        base = np.sum(W_sort[:j]) + 0.5 * W_sort[j]
        weight = 0.5 * W_sort[j] + 0.5 * W_sort[j+1]
        return base + interp_factor * weight

In [85]:
def get_vol_estimator(cdf, n_particles, V_sort):

    U = np.random.rand(n_particles)
    particles = np.zeros(n_particles)

    N = len(V_sort)

    for i, u in enumerate(U):

        j = np.searchsorted(cdf, u)
        j = max(0, min(j, N-2))

        C_prev = 0 if j == 0 else cdf[j-1]
        C_curr = cdf[j]

        v1 = V_sort[j]
        v2 = V_sort[j+1]
        
        if C_curr - C_prev < 1e-15:
            # If the CDF is vertical here, just pick v1
            particles[i] = v1
        else:
            # Linear interpolation
            particles[i] = v1 + (v2 - v1) * (u - C_prev) / (C_curr - C_prev)

    return particles.mean()

In [86]:
def gibbs_sampler(m_b, L_b, a_s, b_s, s_i):

    # initial value
    sigma0 = s_i

    # sample beta | sigma
    cov_b = sigma0**2 * np.linalg.inv(L_b)
    beta_draw = np.random.multivariate_normal(m_b.flatten(), cov_b)
    beta = beta_draw.reshape(-1, 1)

    # sample sigma | beta
    sigma2 = 1 / np.random.gamma(shape=a_s, scale=1/b_s)
    sigma = np.sqrt(sigma2)

    return beta, sigma

In [87]:
def get_rho(phi, omega):
    return phi/np.sqrt(phi**2 + omega)

In [88]:
import numpy as np
from tqdm import tqdm


R = (S / S.shift(1)).dropna().to_numpy().flatten()
n = len(R)

mu_i, kappa_i, theta_i, sigma_i, rho_i = mu0, kappa0, theta0, sigma0, rho0
mu_chain, kappa_chain, theta_chain, sigma_chain, rho_chain = [], [], [], [], []

for i in range(n_samples):
    print(f"\n--- Gibbs Iteration {i+1}/{n_samples} ---")
    
    #Eqn (60)
    V = np.full(n_particles, float(np.atleast_1d(theta_i).item())) 
    v_estimates_list = [] 
    kappa_i, theta_i, sigma_i, mu_i, rho_i = [float(np.atleast_1d(x).item()) for x in [kappa_i, theta_i, sigma_i, mu_i, rho_i]]

    for k in tqdm(range(1, n)):
        R_k = R[k]
        V_prev = np.maximum(V.flatten(), 1e-7) # Floor to prevent sqrt(0)
        
        #Eqn (61)
        epsilon = np.random.normal(0, 1, n_particles)
        #Eqn (62)
        z = np.clip((R_k - mu_i * dt - 1) / (np.sqrt(dt) * np.sqrt(V_prev)), -4.0, 4.0) #Clip Z to prevent extreme returns from blowing up V
        #Eqn (63)
        w = rho_i * z + np.sqrt(1 - rho_i**2) * epsilon
        #Eqn (64)
        V = V_prev + kappa_i * (theta_i - V_prev) * dt + sigma_i * np.sqrt(dt) * np.sqrt(V_prev) * w
        
        # clip particles to a realistic range (max 200% variance)
        V = np.clip(V.flatten(), 1e-8, 2.0)
        #Eqn (65)
        log_W = -0.5 * np.log(2 * np.pi * V * dt + 1e-12) - 0.5 * ((R_k - mu_i * dt - 1)**2) / (V * dt + 1e-12)
        W = np.exp(log_W - np.max(log_W))
        #Eqn (66)
        W /= (np.sum(W) + 1e-15)

        sort_idx = np.argsort(V)
        V_sorted, W_sorted = V[sort_idx], W[sort_idx]
        
        cdf = np.array([convertCDF(v, V_sorted, W_sorted, n_particles) for v in V_sorted])
        v_est = get_vol_estimator(cdf, n_particles, V_sorted)
        
        # Safety check for estimate
        if not np.isfinite(v_est) or v_est <= 0:
            v_est = v_estimates_list[-1] if v_estimates_list else q_i
        v_estimates_list.append(v_est)

    v_estimates_list.append(v_estimates_list[-1])
    v_arr = np.array(v_estimates_list).reshape(-1, 1)
    v_safe = np.maximum(v_arr, 1e-7)

    # estimate mu
    #Eqn (18)
    x_mu = 1.0 / np.sqrt(v_safe * dt)
    #Eqn (17)
    y_mu = R.reshape(-1, 1) / np.sqrt(v_safe * dt)
    #Eqn (19)
    tau_post_mu = (x_mu.T @ x_mu).item() + tau_prior_eta
    #Eqn (20)
    mu_post_eta = ((x_mu.T @ y_mu).item() + mu_prior_eta * tau_prior_eta) / tau_post_mu
    #Eqn (22), (23)
    mu_i = (np.random.normal(mu_post_eta, 1/np.sqrt(tau_post_mu)) - 1) / dt
    mu_chain.append(mu_i)

    # estimate kappa, theta, sigma
    v_l, v_n = v_safe[:-1], v_safe[1:]
    #Eqn (30)
    y_reg = v_n / np.sqrt(v_l * dt)
    #Eqn (31), (32), (34)
    X_reg = np.hstack((1.0 / (v_l * np.sqrt(dt)), np.sqrt(v_l) / np.sqrt(dt)))

    #Eqn (36)
    L_beta_post = X_reg.T @ X_reg + lambda_prior + np.eye(2) * 1e-8
    #Eqn (37)
    mu_beta_post = np.linalg.inv(L_beta_post) @ (lambda_prior @ mu_prior + X_reg.T @ y_reg)
    #Eqn (43)
    a_s_post = a_prior_sigma + n/2
    #Eqn (44)
    quad = (y_reg.T @ y_reg + mu_prior.T @ lambda_prior @ mu_prior - mu_beta_post.T @ L_beta_post @ mu_beta_post).item()
    b_s_post = b_prior_sigma + 0.5 * np.clip(quad, 0, 2.0)          #Clip quadratic form to prevent sigma from exploding

    # Use gibbs sampler since beta and sigma distributions are not independent
    #Eqn (39)
    beta_draw, sigma_i = gibbs_sampler(mu_beta_post, L_beta_post, a_s_post, b_s_post, sigma_i)
    phi_draw = np.clip(beta_draw[1, 0], 0.01, 0.98) # phi is beta[1], < 1 to keep theta and kappa sane.
    #Eqn (40)
    kappa_i = (1.0 - phi_draw) / dt
    
    beta0_draw = np.clip(beta_draw[0, 0], 1e-7, 0.5) # beta[0] is drift, nonnegative
    #Eqn (41)
    theta_i = beta0_draw / (1.0 - phi_draw)
    
    sigma_i = np.clip(sigma_i, 0.01, 1.2)
    kappa_i = np.clip(kappa_i, 0.1, 50.0)
    theta_i = np.clip(theta_i, 1e-5, 0.4)

    sigma_chain.append(sigma_i)
    kappa_chain.append(kappa_i)
    theta_chain.append(theta_i)

    # estimate rho
    #Eqn (45), (46)
    e1 = (R[1:].reshape(-1,1) - mu_i*dt - 1) / (np.sqrt(dt * v_l))
    e2 = (v_n - v_l - kappa_i*(theta_i - v_l)*dt) / (sigma_i * np.sqrt(dt * v_l))

    #Eqn (52), (53)
    A_rho = np.hstack((e1, e2)).T @ np.hstack((e1, e2))
    #Eqn (54), (55)
    mu_phi_post = (A_rho[0, 1] + tau_prior_phi * mu_prior_phi) / (A_rho[0, 0] + tau_prior_phi)
    tau_phi_post = A_rho[0, 0] + tau_prior_phi
    #Eqn (56), (57)
    a_omega_post = a_prior_omega + n/2
    b_omega_post = b_prior_omega + 0.5 * (A_rho[1, 1] - (A_rho[0, 1]**2) / A_rho[0, 0])
    #Eqn (58), (59)
    omega_i = 1 / np.random.gamma(shape=a_omega_post, scale=1/b_omega_post)
    phi_i = np.random.normal(loc=mu_phi_post, scale=np.sqrt(omega_i)/np.sqrt(tau_phi_post))
    #Eqn (48)
    rho_i = np.clip(phi_i / np.sqrt(omega_i + phi_i**2), -0.98, 0.98)
    rho_chain.append(rho_i)


print({
    "mu": np.mean(mu_chain),
    "kappa": np.mean(kappa_chain),
    "theta": np.mean(theta_chain),
    "sigma": np.mean(sigma_chain),
    "rho": np.mean(rho_chain)
})


--- Gibbs Iteration 1/100 ---


100%|██████████| 124/124 [00:01<00:00, 69.65it/s]



--- Gibbs Iteration 2/100 ---


100%|██████████| 124/124 [00:02<00:00, 51.07it/s]



--- Gibbs Iteration 3/100 ---


...


--- Gibbs Iteration 99/100 ---


100%|██████████| 124/124 [00:03<00:00, 39.75it/s]



--- Gibbs Iteration 100/100 ---


100%|██████████| 124/124 [00:02<00:00, 50.45it/s]

{'mu': np.float64(0.2159047693263833), 'kappa': np.float64(12.036917550015087), 'theta': np.float64(0.09440570906210365), 'sigma': np.float64(0.033867421367974444), 'rho': np.float64(0.02471370442178608)}
